# Building an Agent with Long-term Memory using Autogen and Zep

This notebook walks through how to build an Autogen Agent with long-term memory. Zep builds a knowledge graph from user interactions with the agent, enabling the agent to recall relevant facts from previous conversations or user interactions.

In this notebook we will:
- Create an Autogen Agent class that extends `ConversableAgent` by adding long-term memory
- Create a Mental Health Assistant Agent, CareBot, that acts as a counselor and coach.
- Create a user Agent, Cathy, who stands in for our expected user.
- Demonstrate preloading chat history into Zep.
- Demonstrate the agents in conversation, with CareBot recalling facts from previous conversations with Cathy.
- Inspect the knowledge graph Zep builds from the conversation.


Install dependencies:

```bash
pip install pyautogen zep-cloud python-dotenv
```


In [ ]:
import asyncio
import os
import time
import uuid

from dotenv import load_dotenv
from typing import Union, Dict
from autogen import ConversableAgent, Agent

load_dotenv()

config_list = [
    {
        "model": "gpt-5-mini",
        "api_key": os.environ.get("OPENAI_API_KEY"),
        "max_completion_tokens": 1024,
    }
]

## Initialize the Zep Client

You can sign up for a Zep account here: https://www.getzep.com/

In [ ]:
from zep_cloud.client import AsyncZep
from zep_cloud import Message

# Configure Zep
zep = AsyncZep(api_key=os.environ.get("ZEP_API_KEY"))

In [ ]:
def convert_to_zep_messages(chat_history: list[dict[str, str | None]]) -> list[Message]:
    """
    Convert chat history to Zep messages.

    Args:
    chat_history (list): List of dictionaries containing chat messages.

    Returns:
    list: List of Zep Message objects.
    """
    return [
        Message(
            role=msg["role"],
            name=msg.get("name", None),
            content=msg["content"],
        )
        for msg in chat_history
    ]

## ZepConversableAgent

The `ZepConversableAgent` is a custom implementation of the `ConversableAgent` that integrates with Zep for long-term memory management. This class extends the functionality of the base `ConversableAgent` by adding Zep-specific features for persisting and retrieving facts from long-term memory.

In [ ]:
class ZepConversableAgent(ConversableAgent):
    """
    A custom ConversableAgent that integrates with Zep for long-term memory.
    """
    def __init__(
        self,
        name: str,
        system_message: str,
        llm_config: dict,
        function_map: dict,
        human_input_mode: str,
        zep_thread_id: str,
    ):
        super().__init__(
            name=name,
            system_message=system_message,
            llm_config=llm_config,
            function_map=function_map,
            human_input_mode=human_input_mode,
        )
        self.zep_thread_id = zep_thread_id
        # store the original system message as we will update it with relevant facts from Zep
        self.original_system_message = system_message
        self.register_hook(
            "a_process_last_received_message", self.persist_user_messages
        )
        self.register_hook(
            "a_process_message_before_send", self.persist_assistant_messages
        )

    async def persist_assistant_messages(
        self, sender: Agent, message: Union[Dict, str], recipient: Agent, silent: bool
    ):
        """Agent sends a message to the user. Add the message to Zep."""

        # Assume message is a string
        zep_messages = convert_to_zep_messages(
            [{"role": "assistant", "name": self.name, "content": message}]
        )
        await zep.thread.add_messages(self.zep_thread_id, messages=zep_messages)

        return message

    async def persist_user_messages(self, messages: list[dict[str, str]] | str):
        """
        User sends a message to the agent. Add the message to Zep and 
        update the system message with relevant context from Zep.
        """
        # Assume messages is a string
        zep_messages = convert_to_zep_messages([{"role": "user", "content": messages}])
        await zep.thread.add_messages(self.zep_thread_id, messages=zep_messages)

        memory = await zep.thread.get_user_context(thread_id=self.zep_thread_id)

        # Update the system message with the context Zep assembled for this thread
        self.update_system_message(
            self.original_system_message
            + f"\n\nRelevant facts about the user and their prior conversation:\n{memory.context}"
        )

        return messages

## Zep User and Thread Management

### Zep User
A Zep User represents an individual interacting with your application. Each User can have multiple Threads associated with them, allowing you to track and manage interactions over time. The unique identifier for each user is their `UserID`, which can be any string value (e.g., username, email address, or UUID).

### Zep Thread
A Thread represents a conversation and can be associated with Users in a one-to-many relationship. Chat messages are added to Threads, with each thread having many messages.

Zep builds a knowledge graph from those messages, and `thread.get_user_context` returns the context string most relevant to the current conversation. Placing that string in your prompt grounds the agent in what it already knows about the user.

In [ ]:
bot_name = "CareBot"
user_name = "Cathy"

user_id = user_name + str(uuid.uuid4())[:4]
thread_id = str(uuid.uuid4())


await zep.user.add(user_id=user_id)

await zep.thread.create(
    thread_id=thread_id,
    user_id=user_id,
)

## Preload a prior conversation into Zep

We'll load a prior conversation into long-term memory. We'll use facts derived from this conversation when Cathy restarts the conversation with CareBot, ensuring Carebot has context.

In [ ]:
chat_history = [
    {
        "role": "assistant",
        "name": "carebot",
        "content": "Hi Cathy, how are you doing today?",
    },
    {
        "role": "user",
        "name": "Cathy",
        "content": "To be honest, I've been feeling a bit down and demotivated lately. It's been tough.",
    },
    {
        "role": "assistant",
        "name": "CareBot",
        "content": "I'm sorry to hear that you're feeling down and demotivated, Cathy. It's understandable given the challenges you're facing. Can you tell me more about what's been going on?",
    },
    {
        "role": "user",
        "name": "Cathy",
        "content": "Well, I'm really struggling to process the passing of my mother.",
    },
    {
        "role": "assistant",
        "name": "CareBot",
        "content": "I'm deeply sorry for your loss, Cathy. Losing a parent is incredibly difficult. It's normal to struggle with grief, and there's no 'right' way to process it. Would you like to talk about your mother or how you're coping?",
    },
    {
        "role": "user",
        "name": "Cathy",
        "content": "Yes, I'd like to talk about my mother. She was a kind and loving person.",
    },
]

# Convert chat history to Zep messages
zep_messages = convert_to_zep_messages(chat_history)

added = await zep.thread.add_messages(thread_id, messages=zep_messages)

# Ingestion is asynchronous. Poll the returned task so later cells read a
# populated graph instead of an empty one.
async def wait_for_task(task_id, timeout_seconds=120.0, poll_interval_seconds=2.0):
    if not task_id:
        return
    deadline = time.monotonic() + timeout_seconds
    while True:
        task = await zep.task.get(task_id)
        status = (task.status or "").lower()
        if status in {"succeeded", "completed", "complete", "success"}:
            return
        if status in {"failed", "error", "canceled", "cancelled", "partial"}:
            raise RuntimeError(f"Zep task {task_id} ended with status {status}")
        if time.monotonic() >= deadline:
            raise TimeoutError(f"Timed out waiting for Zep task {task_id}")
        await asyncio.sleep(poll_interval_seconds)


await wait_for_task(added.task_id)

## Review what Zep learned

`memory.get_session_facts` was removed in v3. Read the edges of the user's
knowledge graph instead — each edge carries a fact plus its validity window.

In [ ]:
edges = await zep.graph.edge.get_by_user_id(user_id=user_id)

for edge in edges[:10]:
    print(edge.fact)

## Create the Autogen agent, CareBot, an instance of `ZepConversableAgent`

We pass in the current `thread_id` into the CareBot agent which allows it to retrieve relevant context related to the conversation with Cathy.

In [ ]:
carebot_system_message = """
You are a compassionate mental health bot and caregiver. Review information about the user and their prior conversation below and respond accordingly.
Keep responses empathetic and supportive. And remember, always prioritize the user's well-being and mental health. Keep your responses very concise and to the point.
"""

agent = ZepConversableAgent(
    bot_name,
    system_message=carebot_system_message,
    llm_config={"config_list": config_list},
    function_map=None,  # No registered functions, by default it is None.
    human_input_mode="NEVER",  # Never ask for human input.
    zep_thread_id=thread_id,
)

## Create the Autogen agent, Cathy

Cathy is a stand-in for a human. When building a production application, you'd replace Cathy with a human-in-the-loop pattern.

**Note** that we're instructing Cathy to start the conversation with CareBot by asking about her previous conversation. This is an opportunity for us to test whether fact retrieval from Zep's long-term memory is working. 

In [ ]:
cathy = ConversableAgent(
    user_name,
    system_message="You are a helpful mental health bot. You are seeking counsel from a mental health bot. Ask the bot about your previous conversation.",
    llm_config={"config_list": config_list},
    human_input_mode="NEVER",  # Never ask for human input.
)

## Start the conversation

We use Autogen's `a_initiate_chat` method to get the two agents conversing. CareBot is the primary agent.

**NOTE** how Carebot is able to recall the past conversation about Cathy's mother in detail, having had relevant facts from Zep added to its system prompt.

In [ ]:
result = await agent.a_initiate_chat(
    cathy,
    message="Hi Cathy, nice to see you again. How are you doing today?",
    max_turns=3,
)

## Review the graph again

Let's see how the graph has evolved as the conversation has progressed.

In [ ]:
edges = await zep.graph.edge.get_by_user_id(user_id=user_id)

for edge in edges[:10]:
    print(edge.fact)

## Search the knowledge graph

In addition to `thread.get_user_context`, which uses the current conversation to assemble context, we can search the graph with our own query. `graph.search` may be used as an Agent tool, enabling an agent to search across user memory for relevant facts.

In [ ]:
response = await zep.graph.search(
    query="What do you know about Cathy's family?",
    user_id=user_id,
    scope="edges",
    limit=5,
)

for edge in response.edges or []:
    print(edge.fact)